# Resumen de Procesamiento y Modelado de Datos

In [ ]:
import pandas as pd
import numpy as np
import random
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os

## 1. Generación de Datos Finales

In [ ]:
# Glosas de comercios reales, ahora con mayor ambigüedad y solapamiento intencional
# Algunos términos son deliberadamente compartidos entre categorías para introducir confusión
plantillas_reales = {
    "Alimentación": ["JUMBO BILBAO", "LIDER EXPRESS", "MCDONALDS WEB", "ALMACEN VECINAL", "PRONTO COPEC", "PEDIDOSYA CL", "VERDULERIA DON PEPE", "MERCADO CENTRAL", "TIENDA ONLINE", "SUPERMERCADO DIA", "DELIVERY COMIDA"],
    "Transporte": ["COPEC SANTIAGO", "SHELL RUTA 5", "UBER TRIP", "DIDI RIDE", "CARGA BIP WEB", "AUTOPISTA CENTRAL", "TAXI APP", "ESTACIONAMIENTO", "PEAJE TAG", "GASOLINERA"],
    "Salud": ["CRUZ VERDE LOCAL", "FARMACIA AHUMADA", "CLINICA INDISA", "INTEGRAMEDICA", "EXAMENES MED", "CONSULTA ESPECIALISTA", "CENTRO MEDICO", "OPTICA VISION"],
    "Vivienda": ["SODIMAC HOME", "EASY PORTAL", "CONDOMINIO GGCC", "ARRIENDO DEPT", "PINTURAS SHERWIN", "REPARACIONES HOGAR", "SERVICIOS GENERALES", "PAGO CUOTAS", "HOME CENTER"],
    "Educación": ["UDEMY COURSES", "COLEGIO MENSUALIDAD", "MATRICULA UNIV", "SISTEMA DUOC", "LIBRERIA UNIVERSITARIA", "PLATAFORMA APRENDIZAJE", "PAGO CUOTAS", "CURSO IDIOMAS", "TIENDA ONLINE"],
    "Ocio": ["NETFLIX.COM", "SPOTIFY PREMIUM", "CINEMARK ONLINE", "STEAM GAMES", "BAR LA ESQUINA", "RESTAURANT FANCY", "TIENDA ONLINE", "EVENTO DEPORTIVO", "MERCADO CENTRAL", "COMPRA EN AMAZON", "ENTRETENIMIENTO PLUS"],
    "Servicios": ["ENEL DISTRIBUCION", "AGUAS ANDINAS", "VTR BANDA ANCHA", "GASCO LIQUADO", "CLARO TELECOM", "SERVICIO TECNICO", "TALLER AUTOMOTRIZ", "SERVICIOS GENERALES", "PAGO CUOTAS", "TIENDA ONLINE", "BANCO DEL ESTADO"]
}

# Prefijos transaccionales que el banco sí incluye en la glosa de texto
prefijos_banco = ["COMPRA TBK", "TEF ENVIADA", "CARGO TC", "PAGO AUTOMATICO", "PAGO WEB", "DEBITO", "ABONO"]

# Función para simular errores tipográficos ocasionales
def simular_typo(texto, prob_typo=0.05):
    if random.random() < prob_typo:
        # Reemplaza caracteres aleatorios con números o caracteres similares para simular typos
        char_map = {'a': '4', 'e': '3', 'i': '1', 'o': '0', 's': '5', 'l': '1', 'b': '8'}
        mod_texto = list(texto)
        num_cambios = random.randint(1, min(3, len(texto) // 2)) # Hasta 3 cambios o la mitad de la palabra
        for _ in range(num_cambios):
            idx = random.randint(0, len(mod_texto) - 1)
            original_char = mod_texto[idx]
            if original_char.lower() in char_map:
                mod_texto[idx] = char_map[original_char.lower()] if random.random() < 0.7 else original_char.swapcase()
            elif original_char.isalpha():
                mod_texto[idx] = original_char.swapcase()
        return "".join(mod_texto)
    return texto

def generar_dataset_altamente_estructurado_y_ambiguo(num_registros=7000, semilla=42):
    random.seed(semilla)
    datos = []

    for i in range(num_registros):
        cat = random.choice(categorias)

        comercio = random.choice(plantillas_reales[cat])
        prefijo = random.choice(prefijos_banco)

        descripcion_glosa = f"{prefijo} {comercio}"

        # Aplicar typos ocasionales a la glosa
        descripcion_glosa = simular_typo(descripcion_glosa)

        codigo_auth = f"{random.randint(10000000, 99999999)}"

        # Montos (se mantiene la lógica de montos por categoría)
        if cat == "Vivienda": monto = round(random.uniform(50, 400), 2)
        elif cat in ["Alimentación", "Ocio", "Transporte"]: monto = round(random.uniform(5, 60), 2)
        else: monto = round(random.uniform(10, 150), 2)

        datos.append({
            "id_interno": i + 1,
            "codigo_autorizacion": codigo_auth,
            "descripcion": descripcion_glosa,
            "valor": monto,
            "categoria": cat # La categoría real es la principal generada
        })

    return pd.DataFrame(datos)

categorias = ["Alimentación", "Transporte", "Salud", "Vivienda", "Educación", "Ocio", "Servicios"]
df_transacciones = generar_dataset_altamente_estructurado_y_ambiguo(7000, semilla=42)
print("--- Dataset de Transacciones Altamente Ambiguo Generado ---")
print(df_transacciones.head(5))

--- Dataset de Transacciones Altamente Ambiguo Generado ---
   id_interno codigo_autorizacion                 descripcion   valor  \
0           1            42868828  COMPRA TBK SPOTIFY PREMIUM   17.28   
1           2            83197857      DEBITO SPOTIFY PREMIUM    9.78   
2           3            41227216     COMPRA TBK SODIMAC HOME  226.87   
3           4            83140807   TEF ENVIADA TIENDA ONLINE   28.07   
4           5            31429110      ABONO PINTURAS SHERWIN  294.35   

      categoria  
0          Ocio  
1          Ocio  
2      Vivienda  
3  Alimentación  
4      Vivienda  


In [ ]:
def generar_perfiles_volatiles(num_perfiles=5000, semilla=42):
    random.seed(semilla)
    np.random.seed(semilla)

    perfiles = []

    for i in range(num_perfiles):
        # 1. Simular variables base usando distribuciones más realistas
        ingreso = round(np.random.normal(3500, 1500))
        ingreso = max(800, min(ingreso, 9000)) # Forzar límites realistas

        # Nivel de endeudamiento centrado en el 30% (promedio saludable/observación)
        nivel_endeudamiento = np.random.normal(30, 15)
        nivel_endeudamiento = round(max(0, min(nivel_endeudamiento, 85)), 1)

        frecuencia_ahorro = random.choice(["Baja", "Media", "Alta"])

        # Gasto mensual centrado en el 75% del ingreso
        ratio_gasto = np.random.normal(0.75, 0.18)
        ratio_gasto = max(0.25, min(ratio_gasto, 1.3)) # Desde 25% hasta 130% de sobregiro
        gasto_total = round(ingreso * ratio_gasto, 2)

        # Mapeo numérico interno de la conducta de ahorro para la ecuación
        ahorro_val = {"Baja": 0, "Media": 1, "Alta": 2}[frecuencia_ahorro]

        # 2. ECUACIÓN DE NEGOCIO BASE (Lógica financiera pura)
        # El score base sube con la deuda y el gasto, y baja con el ahorro
        score_base = (nivel_endeudamiento * 0.5) + (ratio_gasto * 40) - (ahorro_val * 12)

        # 3. INYECCIÓN DEL RUIDO SANO (Choque Estocástico / Variable No Observada)
        # Una distribución normal con desviación estándar de 10 introduce imprevistos del mundo real.
        # Esto genera los "traslapes" lógicos de los que hablamos.
        choque_imprevisto = np.random.normal(0, 10)
        score_final = score_base + choque_imprevisto

        # 4. Asignación estricta del perfil según el Score Final con Choque incluido
        if score_final >= 52:
            perfil = "En riesgo"
        elif score_final >= 22:
            perfil = "En observación"
        else:
            perfil = "Saludable"

        perfiles.append({
            "id_usuario": i + 1,
            "ingreso_mensual": ingreso,
            "nivel_endeudamiento": nivel_endeudamiento,
            "frecuencia_ahorro": frecuencia_ahorro,
            "gasto_total": gasto_total,
            "ratio_gasto_ingreso": round(ratio_gasto, 2),
            "perfil_financiero": perfil
        })

    return pd.DataFrame(perfiles)

df_perfiles_volatiles = generar_perfiles_volatiles(5000, semilla=42)
print("\n--- Dataset de Perfiles Financieros con Volatilidad Conductual Generado ---")
print(df_perfiles_volatiles.head(5))


--- Dataset de Perfiles Financieros con Volatilidad Conductual Generado ---
   id_usuario  ingreso_mensual  nivel_endeudamiento frecuencia_ahorro  \
0           1             4245                 27.9              Alta   
1           2             3149                 26.5              Baja   
2           3             2796                 38.1              Baja   
3           4             3863                  1.3              Alta   
4           5             1981                 34.7             Media   

   gasto_total  ratio_gasto_ingreso perfil_financiero  
0      3678.65                 0.87    En observación  
1      3256.88                 1.03         En riesgo  
2      1863.77                 0.67    En observación  
3      1697.85                 0.44         Saludable  
4      1161.97                 0.59         Saludable  


## 2. Preprocesamiento NLP y Entrenamiento del Modelo de Transacciones

In [ ]:
# Limpieza básica de texto (la misma función, ya que el preprocesamiento debe ser consistente)
def limpiar_texto(texto):
    texto = texto.lower() # Convertir a minúsculas
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto) # Remover caracteres especiales y números (para enfocarse en el texto)
    return texto

df_transacciones['desc_limpia'] = df_transacciones['descripcion'].apply(limpiar_texto)

# Dividir datos en conjuntos de Entrenamiento y Testeo (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    df_transacciones['desc_limpia'], # Usamos la columna limpia de la descripción
    df_transacciones['categoria'],
    test_size=0.2,
    random_state=42,
    stratify=df_transacciones['categoria'] # Mantiene las clases balanceadas en ambos sets
)

# Convertir el texto a vectores numéricos usando TF-IDF
# Es crucial re-inicializar el vectorizador para que aprenda del nuevo vocabulario ambiguo
vectorizador = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizador.fit_transform(X_train)
X_test_tfidf = vectorizador.transform(X_test)

# Entrenar un clasificador rápido de alto rendimiento (Regresión Logística)
modelo_clasificador = LogisticRegression(max_iter=1000)
modelo_clasificador.fit(X_train_tfidf, y_train)

y_pred = modelo_clasificador.predict(X_test_tfidf)
print("\n--- INFORME DE RENDIMIENTO DEL MODELO DE TRANSACCIONES ---")
print(classification_report(y_test, y_pred))


--- INFORME DE RENDIMIENTO DEL MODELO DE TRANSACCIONES ---
              precision    recall  f1-score   support

Alimentación       0.89      0.87      0.88       196
   Educación       0.82      0.84      0.83       196
        Ocio       0.91      0.90      0.90       197
       Salud       1.00      1.00      1.00       202
   Servicios       0.90      0.78      0.84       204
  Transporte       1.00      1.00      1.00       200
    Vivienda       0.82      0.93      0.87       205

    accuracy                           0.90      1400
   macro avg       0.91      0.90      0.90      1400
weighted avg       0.91      0.90      0.90      1400



## 3. Preprocesamiento y Entrenamiento del Modelo de Perfiles Financieros

In [ ]:
# Codificación de Variables Categóricas (frecuencia_ahorro)
le_volatiles = LabelEncoder()
df_perfiles_volatiles['frecuencia_ahorro_encoded'] = le_volatiles.fit_transform(df_perfiles_volatiles['frecuencia_ahorro'])

# Definición de características (X) y variable objetivo (y)
X_volatiles = df_perfiles_volatiles[['ingreso_mensual', 'nivel_endeudamiento', 'frecuencia_ahorro_encoded', 'gasto_total', 'ratio_gasto_ingreso']]
y_volatiles = df_perfiles_volatiles['perfil_financiero']

# Escalado de Variables Numéricas
scaler_perfiles_volatiles = StandardScaler()
X_scaled_volatiles = scaler_perfiles_volatiles.fit_transform(X_volatiles)
X_scaled_df_volatiles = pd.DataFrame(X_scaled_volatiles, columns=X_volatiles.columns, index=X_volatiles.index)

# División del Dataset
X_train_perfiles_volatiles, X_test_perfiles_volatiles, y_train_perfiles_volatiles, y_test_perfiles_volatiles = train_test_split(
    X_scaled_df_volatiles,
    y_volatiles,
    test_size=0.2,
    random_state=42,
    stratify=y_volatiles
)

# Entrenamiento del Modelo de Clasificación (RandomForestClassifier)
modelo_perfiles_rf_volatiles = RandomForestClassifier(
    n_estimators=200,
    max_depth=7,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
modelo_perfiles_rf_volatiles.fit(X_train_perfiles_volatiles, y_train_perfiles_volatiles)

y_pred_rf_volatiles = modelo_perfiles_rf_volatiles.predict(X_test_perfiles_volatiles)
print("\n--- INFORME DE RENDIMIENTO DEL MODELO DE PERFILES FINANCIEROS (Random Forest) ---")
print(classification_report(y_test_perfiles_volatiles, y_pred_rf_volatiles))


--- INFORME DE RENDIMIENTO DEL MODELO DE PERFILES FINANCIEROS (Random Forest) ---
                precision    recall  f1-score   support

En observación       0.80      0.64      0.71       599
     En riesgo       0.48      0.72      0.58       134
     Saludable       0.64      0.76      0.69       267

      accuracy                           0.68      1000
     macro avg       0.64      0.71      0.66      1000
  weighted avg       0.71      0.68      0.69      1000



## 4. Serialización de Modelos y Preprocesadores

In [ ]:
output_dir = "modelos_y_preprocesadores"
os.makedirs(output_dir, exist_ok=True)

print("📦 Serializando artefactos de Machine Learning...")

joblib.dump(vectorizador, os.path.join(output_dir, 'tfidf_vectorizer.pkl'))
print(f"Vectorizador TF-IDF guardado en: {os.path.join(output_dir, 'tfidf_vectorizer.pkl')}")

joblib.dump(modelo_clasificador, os.path.join(output_dir, 'modelo_clasificador_transacciones.pkl'))
print(f"Modelo de clasificador de transacciones guardado en: {os.path.join(output_dir, 'modelo_clasificador_transacciones.pkl')}")

joblib.dump(scaler_perfiles_volatiles, os.path.join(output_dir, 'scaler_perfiles_volatiles.pkl'))
print(f"Escalador de perfiles volátiles guardado en: {os.path.join(output_dir, 'scaler_perfiles_volatiles.pkl')}")

joblib.dump(le_volatiles, os.path.join(output_dir, 'label_encoder_frecuencia_ahorro_volatiles.pkl'))
print(f"LabelEncoder de frecuencia de ahorro volátiles guardado en: {os.path.join(output_dir, 'label_encoder_frecuencia_ahorro_volatiles.pkl')}")

joblib.dump(modelo_perfiles_rf_volatiles, os.path.join(output_dir, 'modelo_clasificador_perfiles_rf_volatiles.pkl'))
print(f"Modelo de clasificador de perfiles financieros RF volátiles guardado en: {os.path.join(output_dir, 'modelo_clasificador_perfiles_rf_volatiles.pkl')}")

print("\n¡Todos los modelos y preprocesadores han sido serializados exitosamente!")

📦 Serializando artefactos de Machine Learning...
Vectorizador TF-IDF guardado en: modelos_y_preprocesadores/tfidf_vectorizer.pkl
Modelo de clasificador de transacciones guardado en: modelos_y_preprocesadores/modelo_clasificador_transacciones.pkl
Escalador de perfiles volátiles guardado en: modelos_y_preprocesadores/scaler_perfiles_volatiles.pkl
LabelEncoder de frecuencia de ahorro volátiles guardado en: modelos_y_preprocesadores/label_encoder_frecuencia_ahorro_volatiles.pkl
Modelo de clasificador de perfiles financieros RF volátiles guardado en: modelos_y_preprocesadores/modelo_clasificador_perfiles_rf_volatiles.pkl

¡Todos los modelos y preprocesadores han sido serializados exitosamente!


## 5. Prueba de Inferencia de Extremo a Extremo

In [ ]:
output_dir = "modelos_y_preprocesadores"

# 1. Cargar artefactos desde el disco (Como lo haría la API)
vectorizador_loaded = joblib.load(os.path.join(output_dir, 'tfidf_vectorizer.pkl'))
modelo_clasificador_loaded = joblib.load(os.path.join(output_dir, 'modelo_clasificador_transacciones.pkl'))

risk_service = joblib.load(os.path.join(output_dir, 'modelo_clasificador_perfiles_rf_volatiles.pkl'))
encoder_ahorro_service = joblib.load(os.path.join(output_dir, 'label_encoder_frecuencia_ahorro_volatiles.pkl'))
scaler_perfiles_service = joblib.load(os.path.join(output_dir, 'scaler_perfiles_volatiles.pkl'))

# Función de limpieza de texto (debe ser la misma usada en el entrenamiento)
def limpiar_texto_inference(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)
    return texto

# --- PRUEBA 1: Clasificación de una Glosa de Banco (NLP) ---
nueva_glosa = ["COMPRA TBK JUMBO BILBAO STGO"]
nueva_glosa_limpia = limpiar_texto_inference(nueva_glosa[0])
nueva_glosa_tfidf = vectorizador_loaded.transform([nueva_glosa_limpia])
categoria_predicha = modelo_clasificador_loaded.predict(nueva_glosa_tfidf)[0]
print(f"💳 Glosa recibida: '{nueva_glosa[0]}'")
print(f"🏷️  Categoría asignada por Modelo 1: {categoria_predicha}\n")

# --- PRUEBA 2: Evaluación de Riesgo Financiero (Random Forest) ---
nuevo_usuario_json = {
    "ingreso_mensual": 4200,
    "nivel_endeudamiento": 55.0,
    "frecuencia_ahorro": "Baja",
    "gasto_total": 3800.0
}

ratio_gasto = nuevo_usuario_json["gasto_total"] / nuevo_usuario_json["ingreso_mensual"]
ahorro_cod = encoder_ahorro_service.transform([nuevo_usuario_json["frecuencia_ahorro"]])[0]

df_input = pd.DataFrame([{
    'ingreso_mensual': nuevo_usuario_json["ingreso_mensual"],
    'nivel_endeudamiento': nuevo_usuario_json["nivel_endeudamiento"],
    'frecuencia_ahorro_encoded': ahorro_cod,
    'gasto_total': nuevo_usuario_json["gasto_total"],
    'ratio_gasto_ingreso': ratio_gasto
}])

df_input_scaled = scaler_perfiles_service.transform(df_input)
df_input_scaled = pd.DataFrame(df_input_scaled, columns=df_input.columns)

perfil_predicho = risk_service.predict(df_input_scaled)[0]
probabilidades = risk_service.predict_proba(df_input_scaled)[0]

print(f"👤 Evaluación de usuario JSON: {nuevo_usuario_json}")
print(f"🚨 Perfil Asignado por Modelo 2: {perfil_predicho}")
print(f"📊 Distribución de Probabilidades:")
for clase, prob in zip(risk_service.classes_, probabilidades):
    print(f"   - {clase}: {prob*100:.1f}%")

💳 Glosa recibida: 'COMPRA TBK JUMBO BILBAO STGO'
🏷️  Categoría asignada por Modelo 1: Alimentación

👤 Evaluación de usuario JSON: {'ingreso_mensual': 4200, 'nivel_endeudamiento': 55.0, 'frecuencia_ahorro': 'Baja', 'gasto_total': 3800.0}
🚨 Perfil Asignado por Modelo 2: En riesgo
📊 Distribución de Probabilidades:
   - En observación: 7.5%
   - En riesgo: 92.5%
   - Saludable: 0.1%
